# 03 -- Phase 2: Integration & Astrometry

Group calibrated frames, align them (`astroalign`), inverse-variance stack
them, and plate-solve the master with Astrometry.net. The ERR plane is warped
and combined alongside the data; the master gets a WCS.

In [1]:
# ============================================================
#  WORKSHOP CONFIG
# ============================================================
# One shared module rather than this cell copied into six notebooks, so a
# path is changed once and the notebooks cannot drift apart.
# Override any path with an environment variable; see workshop_config.py.
import importlib, os, sys

_here = os.path.dirname(os.path.abspath('workshop_config.py'))
if _here not in sys.path:
    sys.path.insert(0, _here)

# Reloaded, not merely imported. A kernel that imported workshop_config before
# the file was edited keeps serving the cached module, and the first name added
# since then fails much further down as a bare NameError -- which is exactly how
# `raw_frames()` broke for anyone whose kernel predated it.
import workshop_config
importlib.reload(workshop_config)
from workshop_config import *   # noqa: F403  (RAW_DIR, WORK_DIR, PHASE*_DIR, ...)

require_dataset()   # fails now, with the command that fixes it, not later
os.makedirs(WORK_DIR, exist_ok=True)
show_config()

cassa-photometry 0.2.0.dev0
python           3.10.20
solve-field      /media/plato/imtiaz/miniconda3/envs/image_processing/bin/solve-field

  [ok ] workshop  /media/plato/imtiaz/image_processing/cassa_observatory/photometric_pipeline/workshop
  [ok ] raw       /media/plato/imtiaz/image_processing/cassa_observatory/photometric_pipeline/workshop/raw
  [ok ] work      /media/plato/imtiaz/image_processing/cassa_observatory/photometric_pipeline/workshop/work
  [-- ] truth     /media/plato/imtiaz/image_processing/cassa_observatory/photometric_pipeline/workshop/truth_sources.csv
  [ok ] indexes   /media/plato/imtiaz/image_processing/cassa_observatory/photometric_pipeline/astrometry_data

raw tree (106 frames):
    20260903/BIAS/untargeted                   15
    20260903/DARK/untargeted                   15
    20260903/FLAT/untargeted                   56
    20260903/LIGHT/m22                         20

Run `cassa-doctor` for a full environment check.


## Run Phase 2
Uses the shared **local** astrometry index directory set in the config cell
(`CASSA_ASTROMETRY_INDEX`) -- plate solving needs no network. Reads `PHASE1_DIR`
and writes the master stacks into `PHASE2_DIR`.

In [2]:
from cassa_photometry.config import load_config
from cassa_photometry.phase2_integration.pipeline import IntegrationPipeline
cfg = load_config()
pipe = IntegrationPipeline(PHASE1_DIR, output_dir=PHASE2_DIR, config=cfg)
pipe.setup()
pipe.execute()
run_dir = pipe.run_dir
print('Phase 2 output:', run_dir)

10:26:30 [INFO] Hardware: 64 cores, 184.3 GB RAM
10:26:30 [INFO] Not attached to a terminal; using the default CPU level (50%). Pass cpu_level to choose another.
10:26:30 [INFO] Using 32 of 64 cores (50%).

[~] Skimming headers and file sizes...
10:26:30 [INFO] Resource estimate: 20 frames, largest stack 5, peak RAM ~2.2 GB, runtime ~0m 24s.
10:26:30 [INFO] Phase 2 output -> /media/plato/imtiaz/image_processing/cassa_observatory/photometric_pipeline/workshop/work/phase2
10:26:30 [INFO] ==================================================
10:26:30 [INFO] PHASE 2: UNIFIED PIPELINE (Stacks + WCS + QA) - Cores: 32
10:26:30 [INFO] Instrument profile: Generic Instrument
10:26:30 [INFO] ==================================================
10:26:30 [INFO] Plate solver: solve-field
10:26:30 [INFO] 
--- Stacking: M22_Red_CCDSimulator_60s_N61285 (5 frames) ---
10:26:33 [INFO] [+] Anchor set: calibrated_20260903T044211.83935_m22_Red_LIGHT_iub-dhaka_20260903T044211_839350.fits (SNR 8, FWHM unknown)
10:

10:27:00 [INFO]   [*] Initiating Astrometry WCS Solve...
10:27:01 [INFO]     -> Solving with 4 index file(s); SolveHints(ra=279.182375, dec=-23.903388888888887, scale=1.192+/-25%)
10:27:05 [INFO]     [+] Astrometric RMS: 0.753" RA, 0.858" Dec, 1.142" total (48 stars)
10:27:05 [INFO]     [+] SUCCESS: WCS mapped (solve-field).
10:27:05 [INFO] 
--- Stacking: M22_L_CCDSimulator_60s_N61285 (5 frames) ---
10:27:08 [INFO] [+] Anchor set: calibrated_20260903T044111.081822_m22_Luminance_LIGHT_iub-dhaka_20260903T044111_081822.fits (SNR 9, FWHM 4.96 px)
10:27:10 [ERROR]   [-] Alignment failed for calibrated_20260903T043707.171323_m22_Luminance_LIGHT_iub-dhaka_20260903T043707_171323.fits: Input type for source not supported.
10:27:12 [ERROR]   [-] Alignment failed for calibrated_20260903T043910.598881_m22_Luminance_LIGHT_iub-dhaka_20260903T043910_598881.fits: Input type for source not supported.
10:27:14 [ERROR]   [-] Alignment failed for calibrated_20260903T043810.347169_m22_Luminance_LIGHT_iub-d

## Inspect the master stack + its WCS

In [3]:
import glob, os, numpy as np
from astropy.wcs import WCS
from cassa_photometry.fits_utils import read_mef
masters = sorted(glob.glob(os.path.join(run_dir, 'Master_*.fits')))
sci, err, dq, hdr = read_mef(masters[0])
print('STACKCNT:', hdr.get('STACKCNT'), ' TOT_EXP:', hdr.get('TOT_EXP'))
w = WCS(hdr)
print('WCS celestial:', w.has_celestial)
ny, nx = sci.shape
print('Field centre:', w.pixel_to_world(nx/2, ny/2).to_string('hmsdms'))

STACKCNT: 3  TOT_EXP: 180
WCS celestial: True
Field centre: 18h36m43.67942425s -23d54m10.74986864s


### Exercise 1 -- how much deeper is the stack?

Compare the median `ERR` of the master with that of one calibrated frame in the
same filter. A stack of $N$ frames should be about $\sqrt{N}$ deeper.

| Find | Expected |
|---|---|
| depth gain, `ERR`(single) / `ERR`(master) | `TBD` |
| frames stacked, $N$ | `TBD` |

_Expected values come from the reference reduction; `TBD` until that run is fixed._

In [ ]:
from cassa_photometry.fits_utils import read_mef

# Fill in the blanks marked TODO. Everything else is scaffolding.
N = hdr.get('STACKCNT')
filt = hdr.get('FILTER')

# One calibrated frame in the same filter, so the comparison is like for like.
single_path = None
for f in sorted(glob.glob(os.path.join(PHASE1_DIR, 'calibrated_*.fits'))):
    if read_mef(f)[3].get('FILTER') == filt:
        single_path = f
        break
_, single_err, _, _ = read_mef(single_path)

med_single = float(np.nanmedian(single_err))
med_master = FILL_IN     # TODO 1: the same statistic for the master's ERR plane
depth_gain = FILL_IN     # TODO 2: how many times deeper the stack is

print(f"filter {filt}, N = {N} frames stacked\n")
print(f"median ERR, single frame : {med_single:.3f} e-")
print(f"median ERR, master       : {med_master:.3f} e-")
print(f"depth gain               : {depth_gain:.3f}   (sqrt(N) = {np.sqrt(N):.3f})")

### Exercise 2 -- what did the plate solve buy us?

Read the plate scale out of the WCS and work out the field of view.

| Find | Expected |
|---|---|
| plate scale from the WCS | `TBD` arcsec/pixel |
| field of view | `TBD` arcmin |

In [ ]:
from astropy.wcs.utils import proj_plane_pixel_scales

# Fill in the blanks marked TODO. Everything else is scaffolding.
# The WCS was measured against real stars, so it is the authority on the scale --
# not SECPIX, which is whatever the acquisition software was told to write.
scale = FILL_IN          # TODO 1: arcsec/pixel from the WCS (proj_plane_pixel_scales gives deg)
fov_x = FILL_IN          # TODO 2: field of view along x, in arcmin

print(f"plate scale, WCS    : {scale[0]:.4f} x {scale[1]:.4f} arcsec/pixel")
print(f"plate scale, SECPIX : {hdr.get('SECPIX')} arcsec/pixel  (what the camera claimed)")
print(f"image size          : {nx} x {ny} pixels")
print(f"field of view       : {fov_x:.2f} x {ny * scale[1] / 60.0:.2f} arcmin")